In [0]:
bronze_count = spark.table(
    "products_catalog.default.bronze_products"
).count()

silver_count = spark.table(
    "products_catalog.default.silver_products"
).count()

gold_count = spark.table(
    "products_catalog.default.gold_products"
).count()

print("Bronze count:", bronze_count)
print("Silver count:", silver_count)
print("Gold count:", gold_count)

In [0]:
from pyspark.sql import functions as F
gold_df = spark.table(
    "products_catalog.default.gold_products"
)

duplicate_products = (
    gold_df
    .groupBy("product_id")
    .count()
    .filter(
        F.col("count") > 1
    )
)

display(duplicate_products)

In [0]:
from pyspark.sql import functions as F
null_check = gold_df.select(
    F.sum(
        F.when(
            F.col("product_id").isNull(),
            1
        ).otherwise(0)
    ).alias("null_product_id"),

    F.sum(
        F.when(
            F.col("title").isNull(),
            1
        ).otherwise(0)
    ).alias("null_title"),

    F.sum(
        F.when(
            F.col("price").isNull(),
            1
        ).otherwise(0)
    ).alias("null_price"),

    F.sum(
        F.when(
            F.col("category").isNull(),
            1
        ).otherwise(0)
    ).alias("null_category"),

    F.sum(
        F.when(
            F.col("stock").isNull(),
            1
        ).otherwise(0)
    ).alias("null_stock")
)

display(null_check)

In [0]:
from pyspark.sql import functions as F
invalid_prices = gold_df.filter(
    (F.col("price") < 0) |
    (F.col("final_price") < 0) |
    (F.col("discount_percentage") < 0) |
    (F.col("discount_percentage") > 100)
)

print(
    "Invalid price records:",
    invalid_prices.count()
)

display(invalid_prices)

In [0]:
from pyspark.sql import functions as F
invalid_stock = gold_df.filter(
    F.col("stock") < 0
)

print(
    "Invalid stock records:",
    invalid_stock.count()
)

display(invalid_stock)

In [0]:
total_products = gold_df.count()

unique_products = (
    gold_df
    .select("product_id")
    .distinct()
    .count()
)

duplicate_count = (
    total_products - unique_products
)

null_product_id_count = (
    gold_df
    .filter(
        F.col("product_id").isNull()
    )
    .count()
)

invalid_price_count = (
    gold_df
    .filter(
        (F.col("price") < 0) |
        (F.col("final_price") < 0) |
        (F.col("discount_percentage") < 0) |
        (F.col("discount_percentage") > 100)
    )
    .count()
)

invalid_stock_count = (
    gold_df
    .filter(
        F.col("stock") < 0
    )
    .count()
)

if (
    duplicate_count == 0
    and null_product_id_count == 0
    and invalid_price_count == 0
    and invalid_stock_count == 0
):
    validation_status = "PASS"
else:
    validation_status = "FAIL"

print("====================================")
print("       PRODUCT PIPELINE VALIDATION")
print("====================================")
print("Total Products      :", total_products)
print("Unique Products     :", unique_products)
print("Duplicate Products  :", duplicate_count)
print("NULL Product IDs    :", null_product_id_count)
print("Invalid Prices      :", invalid_price_count)
print("Invalid Stock       :", invalid_stock_count)
print("Validation Status   :", validation_status)
print("====================================")